# Posterior Merge of Municipal Outputs

This notebook merges municipal outputs by theme for parquet and pmtiles only.

- Source layout: `Dados/Saída/{theme}/features_{cd_mun}.parquet`
- Merged layout: `Dados/Saída/merged/{theme}/features_merged.parquet|pmtiles`
- `merge_scope="selected"` uses `selected_cd_mun`
- `merge_scope="all"` discovers all municipal files found per theme

In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd
import geopandas as gpd

In [ ]:
base_path = Path("Dados/Saída")
merged_root = base_path / "merged"

merge_scope = "selected"  # "selected" or "all"
selected_cd_mun = ["3550308", "3304557"]

themes = [
    "amenity",
    "leisure",
    "shop",
    "landuse",
    "railway",
    "public_transport",
    "highway",
    "water",
    "bike_tags",
    "footway_tags",
]

In [ ]:
def _normalize_cd_mun(values: List[str]) -> List[str]:
    return [str(v).strip() for v in values if str(v).strip()]


def _discover_theme_parquets(theme: str, scope: str, selected: List[str]) -> List[Path]:
    theme_dir = base_path / theme
    if not theme_dir.exists():
        return []

    if scope == "selected":
        cd_list = set(_normalize_cd_mun(selected))
        return sorted(
            [p for p in theme_dir.glob("features_*.parquet") if p.stem.replace("features_", "") in cd_list]
        )

    if scope == "all":
        return sorted(theme_dir.glob("features_*.parquet"))

    raise ValueError("merge_scope must be 'selected' or 'all'.")


def _merge_theme_parquets(parquet_paths: List[Path]) -> gpd.GeoDataFrame:
    gdfs: List[gpd.GeoDataFrame] = [gpd.read_parquet(p) for p in parquet_paths]
    if not gdfs:
        return gpd.GeoDataFrame()

    merged_df = pd.concat(gdfs, ignore_index=True, sort=False)
    merged_gdf = gpd.GeoDataFrame(merged_df, geometry="geometry", crs=gdfs[0].crs)
    return merged_gdf


def _write_merged_outputs(theme: str, merged_gdf: gpd.GeoDataFrame) -> Tuple[Path, Path]:
    out_dir = merged_root / theme
    out_dir.mkdir(parents=True, exist_ok=True)

    parquet_out = out_dir / "features_merged.parquet"
    pmtiles_out = out_dir / "features_merged.pmtiles"

    merged_gdf.to_parquet(parquet_out, compression="snappy")

    if pmtiles_out.exists():
        pmtiles_out.unlink()

    merged_gdf.to_file(
        pmtiles_out,
        driver="PMTiles",
        engine="pyogrio",
        encoding="utf-8",
        MINZOOM=0,
        MAXZOOM=14,
        NAME=f"layer_{theme}_merged"
    )
    return parquet_out, pmtiles_out

In [ ]:
summary_rows = []

for theme in themes:
    parquet_inputs = _discover_theme_parquets(theme, merge_scope, selected_cd_mun)
    if not parquet_inputs:
        summary_rows.append({
            "theme": theme,
            "municipal_files": 0,
            "parquet": None,
            "pmtiles": None,
            "status": "no_input_files"
        })
        continue

    try:
        merged = _merge_theme_parquets(parquet_inputs)
        if merged.empty:
            summary_rows.append({
                "theme": theme,
                "municipal_files": len(parquet_inputs),
                "parquet": None,
                "pmtiles": None,
                "status": "empty_merged"
            })
            continue

        parquet_out, pmtiles_out = _write_merged_outputs(theme, merged)
        summary_rows.append({
            "theme": theme,
            "municipal_files": len(parquet_inputs),
            "parquet": str(parquet_out),
            "pmtiles": str(pmtiles_out),
            "status": "ok"
        })
    except Exception as exc:
        summary_rows.append({
            "theme": theme,
            "municipal_files": len(parquet_inputs),
            "parquet": None,
            "pmtiles": None,
            "status": f"error: {exc}"
        })

pd.DataFrame(summary_rows)